# Predicting Effluent COD in a Wastewater Treatment Plant

Regression project using the UCI "Water Treatment Plant" dataset and scikit-learn.

## 1. Problem

Wastewater treatment plants measure the water at different stages (plant inlet, primary settler, secondary settler and outlet). Some lab measurements, like chemical oxygen demand (COD), are slow to obtain, so a model that estimates the COD of the water leaving the plant (`DQO-S`) from measurements taken earlier in the process could be useful for early warning and monitoring.

This is a regression problem because the target is a continuous number.

The data is the UCI Water Treatment Plant dataset (https://archive.ics.uci.edu/dataset/106/water+treatment+plant): 527 daily records from one plant in Catalonia, 1990 to 1991. The source does not give units, so errors are in "dataset units". The dataset was originally meant for classification/clustering of plant states, so using it for regression is a different task from its original purpose.

Performance is measured with MAE and R² (RMSE is reported as well). A model is only considered useful if it beats a baseline that always predicts the mean of the training data.

In [ ]:
import warnings, io, zipfile, urllib.request, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

DATA_DIR = Path('data')
MODEL_DIR = Path('models')
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

## 2. Loading the data

Missing values are written as `?` in the file, so `?` is converted to NaN when the file is read. The rows are also not in date order, so the data is sorted by date.

In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/106/water+treatment+plant.zip'
DATA_FILE = DATA_DIR / 'water-treatment.data'

# download the data if it is not already in the data folder
if not DATA_FILE.exists():
    with urllib.request.urlopen(URL, timeout=30) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)

# column names from water-treatment.names
# E = plant inlet, P = primary settler, D = secondary settler inlet, S = outlet, RD = removal performance
COLS = ['Q-E','ZN-E','PH-E','DBO-E','DQO-E','SS-E','SSV-E','SED-E','COND-E',
        'PH-P','DBO-P','SS-P','SSV-P','SED-P','COND-P',
        'PH-D','DBO-D','DQO-D','SS-D','SSV-D','SED-D','COND-D',
        'PH-S','DBO-S','DQO-S','SS-S','SSV-S','SED-S','COND-S',
        'RD-DBO-P','RD-SS-P','RD-SED-P','RD-DBO-S','RD-DQO-S','RD-DBO-G','RD-DQO-G','RD-SS-G','RD-SED-G']

df = pd.read_csv(DATA_FILE, header=None, names=['date'] + COLS, na_values='?')
df['date'] = pd.to_datetime(df['date'].str.replace('D-', '', regex=False), format='%d/%m/%y')
df = df.sort_values('date').reset_index(drop=True)

print(df.shape)
print(df['date'].min().date(), 'to', df['date'].max().date())
df.head()

The file has 527 rows and 38 numeric columns plus the date, and the data covers 1 Jan 1990 to 30 Oct 1991.

## 3. Exploring the data

### 3.1 Missing values

In [ ]:
miss_pct = (df[COLS].isna().mean() * 100).sort_values(ascending=False)
print(miss_pct.head(8).round(1))
print('rows with at least one missing value:', int(df[COLS].isna().any(axis=1).sum()), 'of', len(df))

Not much is missing in the input columns (the biggest is `DBO-P` at around 8%), so no columns are dropped. The gaps are filled with the median inside the model pipeline, so that the median is learned from training data only.

### 3.2 The target (`DQO-S`)

In [ ]:
t = df['DQO-S'].dropna()
print(t.describe().round(1))
print('skewness:', round(t.skew(), 2), '| after log1p:', round(np.log1p(t).skew(), 2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(t, kde=True, ax=ax[0]); ax[0].set_title('DQO-S')
sns.histplot(np.log1p(t), kde=True, ax=ax[1]); ax[1].set_title('log(1 + DQO-S)')
plt.tight_layout(); plt.show()

18 days have no `DQO-S` value, so 509 days are usable. The distribution is strongly right skewed (skewness 2.42). Most days are between about 60 and 100, but a few bad days go up to 350. After a log transform it is much more symmetric (skewness -0.32), so predicting the log of the target and converting back is tested during modelling.

### 3.3 The target over time

In [ ]:
d = df[['date', 'DQO-S']].dropna()
plt.figure(figsize=(12, 4))
plt.scatter(d['date'], d['DQO-S'], s=12, alpha=.6, label='daily value')
plt.plot(d['date'], d['DQO-S'].rolling(30, center=True, min_periods=10).median(), color='crimson', lw=2, label='rolling median')
plt.ylabel('DQO-S'); plt.title('Effluent COD over time'); plt.legend()
plt.tight_layout(); plt.show()

print('lag-1 autocorrelation:', round(d['DQO-S'].autocorr(1), 2))

The COD level is not constant. It was higher in early 1990 (quarterly mean about 110) and lower afterwards (about 70 to 95). The very high days are clustered in two periods, Feb to Apr 1990 and July 1991, which the dataset documentation lists as secondary settler problems. So they are real events and not errors.

Today's COD is also related to yesterday's (autocorrelation 0.43). Because of that, a random train/test split would let the model see days right next to the test days, so the data is split by time instead.

### 3.4 Which inputs are related to the target?

Only measurements taken before the outlet are used as inputs (the `-E`, `-P` and `-D` columns), 22 in total.

In [ ]:
CAND = COLS[:22]
corr_target = df[CAND + ['DQO-S']].corr()['DQO-S'].drop('DQO-S').sort_values()

plt.figure(figsize=(7, 6))
corr_target.plot.barh(color='#4f81bd')
plt.xlabel('correlation with DQO-S'); plt.tight_layout(); plt.show()
print(corr_target.sort_values(ascending=False).head(5).round(2))

The correlations are weak. The best one is `DQO-D` (COD entering the secondary settler) with 0.47, then `DQO-E` (0.32) and `DBO-D` (0.29). That makes sense since the COD just before the last stage should say something about the COD after it, but it also means that only modest predictive performance should be expected.

Many inputs are also highly correlated with each other (for example conductivity at the three stages, around 0.95), which is one reason to try Ridge regression.

### 3.5 Why the `RD-*` columns can't be used

The `RD-*` columns are supposed to be removal percentages, so a check was made on whether `RD-DQO-G` is simply calculated from the inlet and outlet COD.

In [ ]:
calc = 100 * (df['DQO-E'] - df['DQO-S']) / df['DQO-E']
ok = calc.notna() & df['RD-DQO-G'].notna()
print('correlation with the formula:', round(calc[ok].corr(df.loc[ok, 'RD-DQO-G']), 5))

The correlation is basically 1, so `RD-DQO-G` is calculated from the target. Using it (or the other `RD-*` columns, or any other outlet column) as an input would be data leakage, so none of them are used.

### 3.6 Checking for bad values

Conductivity is dissolved salts, which should stay about the same between stages, so `COND-D` should be close to `COND-P` and `COND-S` on the same day. Days where this is not the case are listed below.

In [ ]:
nb = df[['COND-P', 'COND-S']].mean(axis=1)
ratio = df['COND-D'] / nb
bad_cond_d = (ratio < 0.5) | (ratio > 2)
df.loc[bad_cond_d, ['date', 'COND-E', 'COND-P', 'COND-D', 'COND-S']]

Three days (5 Aug 1990, 10 Aug 1990 and 8 Oct 1990) have a `COND-D` far from the other stages, so they are treated as recording errors. The other extreme values are kept (storms, high COD days) because they are real events and those are the days a model would need to handle.

## 4. Cleaning

* set the 3 bad `COND-D` values to NaN
* drop the 18 rows with no target value, since there is nothing to learn from or evaluate on
* missing input values stay as NaN and get filled with the median inside the pipeline

In [ ]:
clean = df.copy()
clean.loc[bad_cond_d, 'COND-D'] = np.nan
clean = clean[clean['DQO-S'].notna()].sort_values('date').reset_index(drop=True)

TARGET = 'DQO-S'
FEATURES = CAND
print('rows:', len(clean))
print('missing input cells: %.2f%%' % (clean[FEATURES].isna().mean().mean() * 100))

## 5. Train / validation / test split

Since the data is a time series, it is split in chronological order: first 70% for training, next 15% for validation and the last 15% for testing. The validation set is used to compare models and the test set is only used once at the end.

In [ ]:
n = len(clean)
i_val, i_test = int(0.70 * n), int(0.85 * n)
train_df, val_df, test_df = clean.iloc[:i_val], clean.iloc[i_val:i_test], clean.iloc[i_test:]

for name, part in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f"{name:10s} {len(part)} rows, {part['date'].min().date()} to {part['date'].max().date()}, target mean {part[TARGET].mean():.1f}, std {part[TARGET].std():.1f}")

Xtr, Xva, Xte = train_df[FEATURES], val_df[FEATURES], test_df[FEATURES]
y_tr, y_va, y_te = train_df[TARGET], val_df[TARGET], test_df[TARGET]

The three parts are not the same: the target mean is 89.6 for training, 78.2 for validation and 85.8 for test, and the test period has the July 1991 problem days, so it is the hardest one.

## 6. Baseline

The baseline predicts the mean of the training target for every day.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'MAE': mean_absolute_error(y_true, y_pred), 'RMSE': float(np.sqrt(mse)), 'R2': r2_score(y_true, y_pred)}

baseline = DummyRegressor(strategy='mean').fit(Xtr, y_tr)
base_val = evaluate(y_va, baseline.predict(Xva))
print({k: round(v, 3) for k, v in base_val.items()})

Baseline validation MAE is 20.53 and R² is -0.24. R² is negative because the COD level went down after the training period, so the training mean is too high for the validation days.

## 7. Models

Four regression models are compared: Linear Regression, Ridge, Random Forest and Gradient Boosting. Each one is a pipeline with a median imputer (and a standard scaler for the linear models, since they are sensitive to feature scale). Each model is also tested with and without the log transform of the target.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.base import clone

def make_model(estimator, scale=False, log_target=True):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', estimator))
    pipe = Pipeline(steps)
    if log_target:
        return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1, check_inverse=False)
    return pipe

# name -> (function that builds the model, needs scaling?)
SPECS = {
    'Linear Regression': (lambda: LinearRegression(), True),
    'Ridge': (lambda: Ridge(alpha=10.0), True),
    'Random Forest': (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5, random_state=SEED, n_jobs=-1), False),
    'Gradient Boosting': (lambda: GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, subsample=0.8, random_state=SEED), False),
}

rows = []
for name, (build, scale) in SPECS.items():
    for log in [False, True]:
        m = make_model(build(), scale, log).fit(Xtr, y_tr)
        res_tr, res_va = evaluate(y_tr, m.predict(Xtr)), evaluate(y_va, m.predict(Xva))
        rows.append([name, 'log' if log else 'raw', res_tr['R2'], res_va['MAE'], res_va['R2']])

results = pd.DataFrame(rows, columns=['model', 'target', 'train_R2', 'val_MAE', 'val_R2'])
print(results.round(3).to_string(index=False))
print('baseline: val_MAE = %.2f, val_R2 = %.3f' % (base_val['MAE'], base_val['R2']))

Observations:

* The log target helps all four models. For example Ridge goes from validation R² -0.081 (MAE 19.32) to 0.122 (MAE 16.87). The log target is used from here on.
* The simple linear models do best on validation: Ridge (MAE 16.87, R² 0.122) and Linear Regression (MAE 16.98, R² 0.116). Gradient Boosting (17.54) and Random Forest (17.65) are slightly worse.
* Random Forest and Gradient Boosting fit the training data much better (train R² above 0.5) but not the validation data, so they overfit. The signal in the data is weak and there are only about 350 training rows.
* All of them beat the baseline MAE of 20.53, but not by a lot.

### Inlet-only features vs all 22

Using only the plant inlet columns (the first 9) was also compared with using all 22 features.

In [ ]:
for label, feats in [('inlet only (9)', COLS[:9]), ('all 22', FEATURES)]:
    m = make_model(Ridge(alpha=10.0), True, True).fit(train_df[feats], y_tr)
    print(label, evaluate(y_va, m.predict(val_df[feats])))

The inlet-only version is worse (validation R² -0.114 compared to 0.122), because it leaves out `DQO-D`, which is the best predictor. All 22 features are therefore kept. This means the model works as a soft sensor using measurements from several stages of the plant on the same day.

## 8. Tuning

Ridge (`alpha`) and Random Forest are tuned with cross-validation on the training set only, using `TimeSeriesSplit` so each fold trains on the past and tests on the days after it. The score is MAE.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
P = 'regressor__model__'   # prefix because the model is inside the log-target wrapper

ridge_search = GridSearchCV(make_model(Ridge(alpha=10.0), True, True),
                            {P + 'alpha': [0.1, 1, 3, 10, 30, 100, 300, 1000]},
                            cv=tscv, scoring='neg_mean_absolute_error')
ridge_search.fit(Xtr, y_tr)

rf_search = RandomizedSearchCV(make_model(SPECS['Random Forest'][0](), False, True),
                               {P + 'n_estimators': [100, 200, 300, 400],
                                P + 'max_depth': [3, 5, 8, 10, None],
                                P + 'min_samples_leaf': [3, 5, 10, 20],
                                P + 'max_features': [0.3, 0.5, 0.7, 1.0]},
                               n_iter=20, cv=tscv, scoring='neg_mean_absolute_error', random_state=SEED)
rf_search.fit(Xtr, y_tr)

print('Ridge best params:', ridge_search.best_params_)
print('Random Forest best params:', rf_search.best_params_)

All models (tuned and untuned) are then compared on the validation set, and the one with the lowest MAE is selected.

In [ ]:
candidates = {name: make_model(build(), scale, True) for name, (build, scale) in SPECS.items()}
candidates['Ridge (tuned)'] = ridge_search.best_estimator_
candidates['Random Forest (tuned)'] = rf_search.best_estimator_

val_scores = {}
for name, m in candidates.items():
    if 'tuned' not in name:
        m.fit(Xtr, y_tr)
    val_scores[name] = evaluate(y_va, m.predict(Xva))

val_table = pd.DataFrame(val_scores).T[['MAE', 'R2']].sort_values('MAE')
print(val_table.round(3))

best_name = val_table.index[0]
best_est = clone(candidates[best_name])
print('\nselected:', best_name)

Tuning made the cross-validation error better, but on the validation set it hardly changed anything (Random Forest 17.65 to 17.15, Ridge 16.87 to 16.92). The best settings were also very regularized (shallow trees, a big Ridge penalty), which fits with the weak signal in the data.

The selected model is **Ridge regression (alpha = 10) with the log target**. It has the lowest validation MAE, and it is also the simplest. The tuned Random Forest is almost tied with it.

In [ ]:
sel = clone(best_est).fit(Xtr, y_tr)
pred_va = sel.predict(Xva)

print('train MAE:', round(mean_absolute_error(y_tr, sel.predict(Xtr)), 1), '| validation MAE:', round(mean_absolute_error(y_va, pred_va), 1))

plt.figure(figsize=(5, 5))
plt.scatter(y_va, pred_va, s=18, alpha=.7)
lim = [0, max(y_va.max(), pred_va.max()) * 1.05]
plt.plot(lim, lim, 'k--'); plt.xlim(lim); plt.ylim(lim)
plt.xlabel('actual DQO-S'); plt.ylabel('predicted'); plt.title('Validation: actual vs predicted')
plt.tight_layout(); plt.show()

Training MAE (21.4) is higher than validation MAE (16.9), so there is no overfitting. Validation is just a calmer period. Training R² is only 0.27, so the model is limited by the information in the inputs and not by the model type. In the plot the predictions stay in a narrow band and the high COD days are not predicted.

## 9. Final evaluation on the test set

The chosen model is refit on training + validation data and tested once on the last 15% of days. The baseline is refit on the same data.

In [ ]:
trval_df = pd.concat([train_df, val_df]).reset_index(drop=True)
Xtv, ytv = trval_df[FEATURES], trval_df[TARGET]

final_model = clone(best_est).fit(Xtv, ytv)
pred_te = final_model.predict(Xte)
base_te = DummyRegressor(strategy='mean').fit(Xtv, ytv).predict(Xte)

TEST_METRICS = evaluate(y_te, pred_te)
BASE_METRICS = evaluate(y_te, base_te)
print('Final model:', {k: round(v, 3) for k, v in TEST_METRICS.items()})
print('Baseline   :', {k: round(v, 3) for k, v in BASE_METRICS.items()})

plt.figure(figsize=(12, 4))
plt.plot(test_df['date'], y_te.values, 'o-', ms=3, lw=.8, label='actual')
plt.plot(test_df['date'], pred_te, 's-', ms=3, lw=.8, label='predicted')
plt.legend(); plt.title('Test period'); plt.ylabel('DQO-S')
plt.tight_layout(); plt.show()

worst = test_df.assign(pred=pred_te, abs_err=np.abs(y_te.values - pred_te)).nlargest(5, 'abs_err')
print(worst[['date', TARGET, 'pred', 'abs_err']].round(1))

Test results:

| | Final model (Ridge) | Baseline (mean) |
|---|---|---|
| MAE | 27.85 | 33.36 |
| RMSE | 46.86 | 50.29 |
| R² | 0.131 | -0.001 |

The model beats the baseline on every metric (MAE is 16.5% lower and R² is positive), but the R² of 0.13 is low. The gap between RMSE and MAE shows there are a few very large errors. The five worst days are all in July 1991 (the 17th to 19th had COD of 236 to 292 and the model predicted about 78 to 104). These are the secondary settler problem days, and the upstream measurements don't show them coming.

So the model captures some of the normal day to day variation, but it can't warn about equipment or settler failures. The test set is only 77 days and dominated by one episode, so the result is not very precise.

## 10. Which features matter?

Permutation importance is computed on the validation set (one feature is shuffled and the increase in MAE is measured).

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(sel, Xva, y_va, n_repeats=30, random_state=SEED, scoring='neg_mean_absolute_error')
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values().tail(10)

plt.figure(figsize=(7, 4.5))
imp.plot.barh(color='#4f81bd')
plt.xlabel('increase in validation MAE when shuffled'); plt.title('Permutation importance')
plt.tight_layout(); plt.show()

`DQO-D` is by far the most important feature (shuffling it raises the MAE by about 5.7), followed by `SED-E`, `SED-P` and `SSV-E`. This is reasonable, because the COD entering the last stage should be the main driver of the COD leaving it. Since many features are correlated, individual importances should be interpreted with caution, and importance does not imply causation.

## 11. Saving the model

The whole pipeline (imputer, scaler, model and log transform) is saved so new data can go straight into `.predict()`. The metadata file has the feature order and the observed ranges for the app.

In [ ]:
FEATURE_DESCRIPTIONS = {
 'Q-E': 'input flow to plant', 'ZN-E': 'input zinc to plant', 'PH-E': 'input pH to plant',
 'DBO-E': 'input BOD to plant', 'DQO-E': 'input COD to plant', 'SS-E': 'input suspended solids to plant',
 'SSV-E': 'input volatile suspended solids to plant', 'SED-E': 'input sediments to plant', 'COND-E': 'input conductivity to plant',
 'PH-P': 'input pH to primary settler', 'DBO-P': 'input BOD to primary settler', 'SS-P': 'input suspended solids to primary settler',
 'SSV-P': 'input volatile suspended solids to primary settler', 'SED-P': 'input sediments to primary settler', 'COND-P': 'input conductivity to primary settler',
 'PH-D': 'input pH to secondary settler', 'DBO-D': 'input BOD to secondary settler', 'DQO-D': 'input COD to secondary settler',
 'SS-D': 'input suspended solids to secondary settler', 'SSV-D': 'input volatile suspended solids to secondary settler',
 'SED-D': 'input sediments to secondary settler', 'COND-D': 'input conductivity to secondary settler',
 'DQO-S': 'output COD'}

MODEL_PATH = MODEL_DIR / 'water_treatment_model.joblib'
META_PATH = MODEL_DIR / 'model_metadata.json'
joblib.dump(final_model, MODEL_PATH, compress=3)

feature_info = {}
for f in FEATURES:
    s = trval_df[f].dropna()
    feature_info[f] = {'description': FEATURE_DESCRIPTIONS[f], 'min': float(s.min()), 'p01': float(s.quantile(.01)),
                       'median': float(s.median()), 'p99': float(s.quantile(.99)), 'max': float(s.max())}

meta = {
    'target': TARGET, 'target_description': FEATURE_DESCRIPTIONS[TARGET], 'target_units': 'not documented by the dataset source',
    'model': 'Ridge' if 'Ridge' in best_name else best_name, 'log_target': True,
    'features': FEATURES, 'feature_info': feature_info,
    'trained_on': {'rows': int(len(trval_df)), 'from': str(trval_df['date'].min().date()), 'to': str(trval_df['date'].max().date())},
    'training_target': {'mean': float(ytv.mean()), 'median': float(ytv.median()), 'min': float(ytv.min()), 'max': float(ytv.max())},
    'test_metrics': {k: float(v) for k, v in TEST_METRICS.items()},
    'baseline_test_metrics': {k: float(v) for k, v in BASE_METRICS.items()},
    'test_period': {'rows': int(len(test_df)), 'from': str(test_df['date'].min().date()), 'to': str(test_df['date'].max().date())},
}
with open(META_PATH, 'w') as fh:
    json.dump(meta, fh, indent=2)
print('saved', MODEL_PATH, 'and', META_PATH)

In [ ]:
# quick test: reload the model and predict one day (missing inputs are filled with the training median)
def predict_effluent_cod(sample, model=None):
    model = model if model is not None else joblib.load(MODEL_PATH)
    row = pd.DataFrame([sample]).reindex(columns=FEATURES).astype(float)
    return float(model.predict(row)[0])

reloaded = joblib.load(MODEL_PATH)
print('same predictions as before saving:', bool(np.allclose(reloaded.predict(Xte), pred_te)))

example = test_df.iloc[0][FEATURES].to_dict()
print('prediction:', round(predict_effluent_cod(example, reloaded), 1), '| actual:', test_df.iloc[0][TARGET])
print('with only 3 inputs:', round(predict_effluent_cod({'DQO-E': 450.0, 'DBO-E': 200.0, 'DQO-D': 280.0}, reloaded), 1))

## 12. Deployment

The model is deployed with a Streamlit app (`app.py`). It loads the saved model and metadata, makes one input box for each of the 22 features (with ranges from the training data), lets the user leave a measurement empty (the pipeline fills it with the median), and shows the predicted effluent COD. The app says clearly that the result is only an estimate and doesn't replace lab measurements.

Run it locally with `streamlit run app.py`.

In real use, predictions should be compared with the laboratory results as they arrive, and the model retrained when the error increases. The inputs in the test period were already different from the training period (for example lower inlet flow), so the model would need to be updated over time.

## 13. Conclusion

This project developed a regression model to predict the effluent COD of a wastewater treatment plant from measurements taken earlier in the process. The best model was Ridge regression on the log of the target, which got a test MAE of 27.85 and R² of 0.131, compared to 33.36 and -0.001 for the baseline.

Main findings:

* Effluent COD is only weakly predictable from this data. The model explains about 12 to 13% of the variance, a small but real improvement over the baseline.
* The log transform and splitting by time made more difference than the choice of algorithm. Random Forest and Gradient Boosting did not do better than Ridge.
* The biggest errors are on the July 1991 settler problem days, which can't be predicted from the available measurements.
* The most useful feature is `DQO-D`, the COD entering the secondary settler.

Limitations: only one plant, a short record, small validation and test sets (76 and 77 days), no units in the data and no variables like temperature or weather. The data is also over 30 years old. The model is a prototype and shouldn't replace lab analysis.

Possible improvements: more data from more plants, predicting the risk of a fault day as a classification problem (which is closer to what the dataset was made for), and adding prediction intervals.

## References

* Poch, M. (1993). *Water Treatment Plant* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5FS4C
* Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825-2830.